In [61]:
import sqlite3
conn = sqlite3.connect("Clientes.db")

In [62]:
cur = conn.cursor()

In [64]:
#Lectura del archivo CSV - Archivo "facturas"
import os
import csv

os.chdir("/Users/yair_/Downloads")
with open("facturas.csv") as f:
    reader = csv.reader(f)
    data = list(reader)

In [65]:
#Creación de la tabla donde se importará el archivo csv
cur.execute("""
    CREATE TABLE facturas (
    IdCustomer INTEGER,
    Factura REAL,
    Importe REAL);""")

#Inserción de los datos en la tabla recién creada
for renglon in range(1,len(data)):
    cur.execute("""
        INSERT INTO facturas (IdCustomer, Factura, Importe)
        VALUES (?,?,?)""", data[renglon])

conn.commit()

In [66]:
#Lectura del archivo CSV - Archivo "datos_clientes"
import os
import csv

os.chdir("/Users/yair_/Downloads")
with open("datos_clientes.csv") as f:
    reader = csv.reader(f)
    data = list(reader)

In [67]:
#Creación de la tabla donde se importará el archivo csv
cur.execute("""
    CREATE TABLE datos (
    IdCustomer INTEGER PRIMARY KEY,
    Name TEXT,
    Age INTEGER);""")

#Inserción de los datos en la tabla recién creada
for renglon in range(1,len(data)):
    cur.execute("""
        INSERT INTO datos (IdCustomer, Name, Age)
        VALUES (?,?,?)""", data[renglon])

conn.commit()

In [43]:
#Consulta mediante inner join (Se muestran los Id y nombre de los clientes que tienen facturas registrados)
cur.execute("""SELECT DISTINCT p2.IdCustomer, p1.Name
                FROM datos AS p1 INNER JOIN facturas AS p2
                ON p1.IdCustomer = p2.IdCustomer
                ORDER BY p1.IdCustomer""")
result = cur.fetchall()
result

[(1, 'Carlos Méndez'),
 (3, 'Andrés Ramírez'),
 (4, 'Sofía Morales'),
 (5, 'Javier Ortega'),
 (8, 'Gabriela Vargas'),
 (9, 'Ricardo Domínguez'),
 (12, 'Valeria Pineda'),
 (13, 'Manuel Rojas'),
 (14, 'Camila Navarro'),
 (17, 'Enrique Torres'),
 (18, 'Natalia Guzmán'),
 (19, 'Francisco Ibáñez'),
 (20, 'Lucía Aguilar'),
 (21, 'Tomás Fernández'),
 (22, 'Daniela Cabrera')]

In [47]:
#Consulta mediante left join (Se muestran todos los id y nombres de los clientes aunque no tengan facturas registrados)
cur.execute("""SELECT p1.IdCustomer, p1.Name, p2.Factura
                FROM datos AS p1 LEFT JOIN facturas AS p2
                ON p1.IdCustomer = p2.IdCustomer
                ORDER BY p1.IdCustomer""")
result = cur.fetchall()
result

[(1, 'Carlos Méndez', 6549.0),
 (2, 'Laura Castillo', None),
 (3, 'Andrés Ramírez', 7360.0),
 (3, 'Andrés Ramírez', 9367.0),
 (4, 'Sofía Morales', 466.0),
 (5, 'Javier Ortega', 504.0),
 (6, 'Mariana López', None),
 (7, 'Daniel Herrera', None),
 (8, 'Gabriela Vargas', 5690.0),
 (8, 'Gabriela Vargas', 9329.0),
 (9, 'Ricardo Domínguez', 532.0),
 (9, 'Ricardo Domínguez', 8457.0),
 (10, 'Fernanda Salazar', None),
 (11, 'Alejandro Cruz', None),
 (12, 'Valeria Pineda', 6416.0),
 (13, 'Manuel Rojas', 507.0),
 (14, 'Camila Navarro', 553.0),
 (14, 'Camila Navarro', 723.0),
 (14, 'Camila Navarro', 8218.0),
 (14, 'Camila Navarro', 9590.0),
 (15, 'Luis Serrano', None),
 (16, 'Paola Álvarez', None),
 (17, 'Enrique Torres', 6381.0),
 (17, 'Enrique Torres', 8145.0),
 (18, 'Natalia Guzmán', 563.0),
 (18, 'Natalia Guzmán', 6479.0),
 (19, 'Francisco Ibáñez', 7599.0),
 (19, 'Francisco Ibáñez', 8221.0),
 (20, 'Lucía Aguilar', 7132.0),
 (21, 'Tomás Fernández', 7159.0),
 (22, 'Daniela Cabrera', 480.0),
 (23,

In [70]:
#Uso de CASE WHEN para etiquetar a los clientes en prioridades
cur.execute('''SELECT IdCustomer, SUM(Importe) AS TotalImporte,
               CASE WHEN SUM(Importe) > 5000 Then 'Cliente Premium'
               ELSE 'Cliente General' END AS Category
               FROM facturas 
               GROUP BY IdCustomer
               ORDER BY TotalImporte DESC
               ''')
result = cur.fetchall()
result

[(14, 217867.31, 'Cliente Premium'),
 (9, 27052.22, 'Cliente Premium'),
 (18, 11110.25, 'Cliente Premium'),
 (3, 8533.8, 'Cliente Premium'),
 (5, 6618.51, 'Cliente Premium'),
 (20, 6616.34, 'Cliente Premium'),
 (21, 4830.94, 'Cliente General'),
 (1, 4778.87, 'Cliente General'),
 (17, 4293.4, 'Cliente General'),
 (13, 2903.12, 'Cliente General'),
 (8, 2703.45, 'Cliente General'),
 (12, 1871.45, 'Cliente General'),
 (22, 1736.0, 'Cliente General'),
 (4, 1636.48, 'Cliente General'),
 (19, 1172.9, 'Cliente General')]

In [72]:
#Ejemplo de SubQueries semi-join
cur.execute('''SELECT IdCustomer, Name
               FROM datos
               WHERE IdCustomer IN
                   (SELECT IdCustomer
                    FROM facturas)''')
result = cur.fetchall()
result

[(1, 'Carlos Méndez'),
 (3, 'Andrés Ramírez'),
 (4, 'Sofía Morales'),
 (5, 'Javier Ortega'),
 (8, 'Gabriela Vargas'),
 (9, 'Ricardo Domínguez'),
 (12, 'Valeria Pineda'),
 (13, 'Manuel Rojas'),
 (14, 'Camila Navarro'),
 (17, 'Enrique Torres'),
 (18, 'Natalia Guzmán'),
 (19, 'Francisco Ibáñez'),
 (20, 'Lucía Aguilar'),
 (21, 'Tomás Fernández'),
 (22, 'Daniela Cabrera')]

In [73]:
#Ejemplo de SubQueries anti-join
cur.execute('''SELECT IdCustomer, Name
               FROM datos
               WHERE IdCustomer NOT IN
                   (SELECT IdCustomer
                    FROM facturas)''')
result = cur.fetchall()
result

[(2, 'Laura Castillo'),
 (6, 'Mariana López'),
 (7, 'Daniel Herrera'),
 (10, 'Fernanda Salazar'),
 (11, 'Alejandro Cruz'),
 (15, 'Luis Serrano'),
 (16, 'Paola Álvarez'),
 (23, 'Sergio Molina'),
 (24, 'Elena Duarte')]

In [74]:
#Uso de SubQuery para encontrar el ID del cliente que ha comprado más
cur.execute('''SELECT IdCustomer, Importe
               FROM facturas
               WHERE Importe =
                   (SELECT MAX(Importe) FROM facturas)''')
result = cur.fetchall()
result

[(14, 180323.97)]